In [ ]:
TEST = False

INPUT_FN = "input_test.txt"
TEST_SOLUTION = 357
TEST_SOLUTION2 = 3121910778619

if not TEST:
    INPUT_FN = "input.txt"

def printd(s):
    if TEST:
        print (s)

In [2]:
# with open(INPUT_FN) as f:
#     s = f.read()

# patterns_block, designs_block = s.split('\n\n')

# Read and split lines
with open(INPUT_FN) as f:
    lines = [l.rstrip() for l in f]

if lines[-1]=='':
    lines=lines[:,-1]

In [3]:
lines

['987654321111111', '811111111111119', '234234234234278', '818181911112111']

# Algorithm

Maybe it can be solved by using a double pointer, one coming from the left and another from the right. Only start moving the other side, towards the center if the value decreases.

Test with first line

In [4]:
line0 = lines[2]
line0

'234234234234278'

In [5]:
LEFT,RIGHT = 0,1

max_left = 0
max_right =0

pos_left = 0
pos_right = len(line0)-1

side=LEFT

while pos_left<=pos_right+1:

    nleft = int(line0[pos_left])
    nright = int(line0[pos_right])

    print(f"{pos_left=}, {nleft=}   {pos_right=}, {nright=}    {side=}")
    if side==LEFT:
        if nleft>=max_left:
            max_left=nleft
        else:
            side=RIGHT
        pos_left+=1
    else:
        if nright>=max_right:
            max_right=nright
        else:
            side=LEFT
        pos_right-=1

print(f"{max_left=}, {max_right=}")


pos_left=0, nleft=2   pos_right=14, nright=8    side=0
pos_left=1, nleft=3   pos_right=14, nright=8    side=0
pos_left=2, nleft=4   pos_right=14, nright=8    side=0
pos_left=3, nleft=2   pos_right=14, nright=8    side=0
pos_left=4, nleft=3   pos_right=14, nright=8    side=1
pos_left=4, nleft=3   pos_right=13, nright=7    side=1
pos_left=4, nleft=3   pos_right=12, nright=2    side=0
pos_left=5, nleft=4   pos_right=12, nright=2    side=1
pos_left=5, nleft=4   pos_right=11, nright=4    side=0
pos_left=6, nleft=2   pos_right=11, nright=4    side=0
pos_left=7, nleft=3   pos_right=11, nright=4    side=1
pos_left=7, nleft=3   pos_right=10, nright=3    side=0
pos_left=8, nleft=4   pos_right=10, nright=3    side=1
pos_left=8, nleft=4   pos_right=9, nright=2    side=0
pos_left=9, nleft=2   pos_right=9, nright=2    side=0
pos_left=10, nleft=3   pos_right=9, nright=2    side=1
max_left=4, max_right=8


Doesn't work

# Algorithm 2

Brute force searching almost all combinations

In [6]:
def get_max_joltage(line0):
    max_joltage = 0
    max_joltage_1st_digit = 0
    for i0 in range(len(line0)-1):
        n0 = int(line0[i0])
        if n0<=max_joltage_1st_digit:
            continue #skip
        for i1 in range(i0+1,len(line0)):
            n1 = int(line0[i1])

            candidate = n0*10+n1
            if candidate > max_joltage:
                max_joltage= candidate
                max_joltage_1st_digit = n0

    print(f"{line0=}, {max_joltage}")

    return max_joltage


In [7]:
get_max_joltage(lines[3])

line0='818181911112111', 92


92

works

Calculate solution

In [8]:
sum_joltages = 0

for line0 in lines:
    j0 = get_max_joltage(line0)
    sum_joltages+=j0

line0='987654321111111', 98
line0='811111111111119', 89
line0='234234234234278', 78
line0='818181911112111', 92


In [9]:
if TEST:
    assert sum_joltages == TEST_SOLUTION

In [10]:
sum_joltages

357

# Part2

12 digits

Probably best to solve as a recursion

In [11]:
def get_max_jolt_ndigits( iter, bank_rem, ndigits_rem ):
    # returns a value of the max jolt
    # I am not sure it is working. But it is very slow for large numbers
    #bank_rem as a string
    #global max_jolt_list

    printd(f"get_max_jolt_ndigits : {iter=}, {bank_rem=}, {ndigits_rem=}")
    
    max_digit=0
    max_value = 0

    if len(bank_rem) < ndigits_rem:
        return 0
    
    for i0 in range(len(bank_rem)-ndigits_rem):
        n0 = int(bank_rem[i0])
        if n0>= max_digit:
            max_digit=n0
            if ndigits_rem==0:
                max_value=max_digit
            else:
                r = get_max_jolt_ndigits(iter+1, bank_rem[i0+1:], ndigits_rem-1)
                value = n0* 10**ndigits_rem + r
                max_value= max(value, max_value)
    
    printd (f"{iter=}, {max_value=}")

    return max_value

In [12]:
# get_max_jolt_ndigits(0, lines[2], 11)

it works with short lengths but not with large lengths

In [13]:
# sum_joltages2 = 0

# for line0 in lines:
#     j0 = get_max_jolt_ndigits(0, line0, 11)
#     sum_joltages2+=j0

In [14]:
# if TEST:
#     assert TEST_SOLUTION2 == sum_joltages2

In [15]:
# sum_joltages2 

Use a different technique to the function above.
First collect digits and their positions as a ordered dict. Reorder according to the digit value and then to the index positions.

Then start with the highest value and its lowest index. Remove numbers that have indices lower than the value selected.
Recursively remove numbers until last

In [30]:
from collections import OrderedDict, defaultdict

dd0 = defaultdict(list)

In [31]:
s0 = lines[2]
print(s0)
for i, c in enumerate(s0):
    dd0[int(c)].append(i)

234234234234278


In [32]:
dd0

defaultdict(list,
            {2: [0, 3, 6, 9, 12],
             3: [1, 4, 7, 10],
             4: [2, 5, 8, 11],
             7: [13],
             8: [14]})

In [33]:
od0 = OrderedDict(sorted(dd0.items(), key = lambda x: x[0] , reverse=True))

In [34]:
od0

OrderedDict([(8, [14]),
             (7, [13]),
             (4, [2, 5, 8, 11]),
             (3, [1, 4, 7, 10]),
             (2, [0, 3, 6, 9, 12])])

In [35]:
def dict_filter_lower_indices(od, index):
    od1 = od.copy()

    for digit, i_list in od.items():
        i_list1 = list(filter(lambda x: x>index, i_list))

        if len(i_list1)==0:
            od1.pop(digit,None) #remove key from dict
        else:
            # no need to sort, it is assumed is already sorted
            od1[digit] = i_list1

    return od1


In [36]:
# try
dict_filter_lower_indices(od0, 70)

OrderedDict()

Recursive function

In [40]:
def get_max_jolt_rec( iter, od_rem, number_string ):
    # returns a value of the max jolt as a list

    # select higher value
    if iter==12:
        print(f"Final number string: {number_string}")
        return number_string
    
    for digit, indices in od_rem.items():
        assert len(indices)>0
        i0 = indices[0]
        
        od1 = dict_filter_lower_indices(od_rem, i0)

        number_string1 = number_string + str(digit)

        res = get_max_jolt_rec(iter+1, od1, number_string1)

        if res is not None:
            return res
    
    return None

In [41]:
get_max_jolt_rec(0, od0, "")

Final number string: 434234234278


'434234234278'

In [42]:
print (s0)

234234234234278


In [43]:
def get_max_jolt2(bank):
    dd0 = defaultdict(list)
    for i, c in enumerate(bank):
        dd0[int(c)].append(i)

    # sort keys descending
    od0 = OrderedDict(sorted(dd0.items(), key = lambda x: x[0] , reverse=True))

    res = get_max_jolt_rec(0, od0, "")

    return res

It looks like it is working

In [44]:
sum_joltages2 = 0

for line0 in lines:
    
    j0 = int(get_max_jolt2(line0))
    sum_joltages2+=j0

Final number string: 987654321111
Final number string: 811111111119
Final number string: 434234234278
Final number string: 888911112111


In [45]:
if TEST:
    assert TEST_SOLUTION2 == sum_joltages2

In [46]:
sum_joltages2 

3121910778619